# ORM VS PRM

# Python模拟一个Agent Loop

[ORM VS PRM](https://walkinglabs.github.io/hands-on-modern-rl/chapter22_agentic/credit-assignment#step-level-advantage-%E7%9A%84%E4%B8%89%E7%B1%BB%E6%9D%A5%E6%BA%90)

## 搭建一个 Mini Tool Environment
我们用纯 Python 搭建一个模拟的“研究助手”环境。Agent 可以调用三种工具：

| 工具 | 功能 | 返回 |
| --- | --- | --- |
| `search(query)` | 模拟搜索信息 | 搜索结果文本 |
| `calculate(expr)` | 执行数学计算 | 计算结果 |
| `verify(fact)` | 验证某个事实 | `True` / `False` |

In [ ]:
# ==========================================
# 1. Mini Tool Environment
# ==========================================
import re
from dataclasses import dataclass

@dataclass
class ToolResult:
    """工具调用的返回结果"""
    tool: str          # 工具名称
    input: str         # 调用输入
    output: str        # 返回内容
    success: bool      # 是否成功

class MiniToolEnv:
    """模拟的轻量工具环境"""

    # 预设的"知识库"——搜索工具会从这里查
    KNOWLEDGE = {
        "earth_radius": "6371",
        "pi": "3.14159265",
        "speed_of_light": "299792458",
        "gravity": "9.8",
        "moon_distance": "384400",
        "population_china": "1400000000",
        "python_release": "1991",
        "gpt_release": "2020",
        "transformer_paper": "2017",
    }

    def search(self, query: str) -> ToolResult:
        """模拟搜索：在预设知识库中查找"""
        query_lower = query.lower()
        for key, value in self.KNOWLEDGE.items():
            if key in query_lower or any(w in key for w in query_lower.split("_")):
                return ToolResult("search", query, f"找到：{key} = {value}", True)
        return ToolResult("search", query, f"未找到与'{query}'相关的信息", False)

    def calculate(self, expression: str) -> ToolResult:
        """模拟计算器：安全的数学表达式求值"""
        try:
            safe_expr = re.sub(r'[^0-9+\-*/().]', '', expression)
            result = eval(safe_expr)
            return ToolResult("calculate", expression, str(result), True)
        except:
            return ToolResult("calculate", expression, "计算错误", False)

    def verify(self, fact: str) -> ToolResult:
        """模拟事实核查"""
        for key, value in self.KNOWLEDGE.items():
            if key in fact.lower() and value in fact:
                return ToolResult("verify", fact, "正确", True)
        return ToolResult("verify", fact, "无法验证", False)

# 测试环境
env = MiniToolEnv()
print(env.search("earth_radius"))
print(env.calculate("2 * 3.14159 * 6371"))
print(env.verify("earth_radius is 6371"))

## 定义Agent Loop

In [ ]:
# ==========================================
# 2. Agent Turn 与 Episode 定义
# ==========================================
@dataclass
class Turn:
    """一个交互轮次"""
    action: str          # "search" | "calculate" | "verify" | "answer"
    input: str           # 工具输入或最终答案
    observation: str     # 环境返回
    success: bool        # 工具调用是否成功

@dataclass
class Episode:
    """一个完整的 Agent 交互过程"""
    task: str
    ground_truth: str
    turns: List[Turn]

def run_agent_loop(env, task, action_plan, ground_truth):
    """执行一次 Agent 交互循环。"""
    turns = []
    for step in action_plan:
        tool = step["tool"]
        inp = step["input"]

        if tool == "search":
            result = env.search(inp)
        elif tool == "calculate":
            result = env.calculate(inp)
        elif tool == "verify":
            result = env.verify(inp)
        elif tool == "answer":
            correct = inp.strip() == ground_truth.strip()
            turns.append(Turn("answer", inp,
                              "正确！" if correct else "错误", correct))
            return Episode(task, ground_truth, turns)
        else:
            result = ToolResult(tool, inp, f"未知工具: {tool}", False)

        turns.append(Turn(tool, inp, result.output, result.success))

    return Episode(task, ground_truth, turns)

## 设计一个多步任务
任务："地球的赤道周长是多少公里？" 正确路径是 search → calculate → verify → answer。

In [ ]:
# 正确的工具调用序列
good_plan = [
    {"tool": "search", "input": "earth_radius"},
    {"tool": "calculate", "input": "2 * 3.14159 * 6371"},
    {"tool": "verify", "input": "earth_radius is 6371"},
    {"tool": "answer", "input": "40030"},
]

# 第 2 步算错了（π 取成了 3）
bad_plan = [
    {"tool": "search", "input": "earth_radius"},
    {"tool": "calculate", "input": "2 * 3 * 6371"},
    {"tool": "verify", "input": "earth_radius is 6371"},
    {"tool": "answer", "input": "38226"},
]

good_episode = run_agent_loop(env, task, good_plan, ground_truth)
bad_episode = run_agent_loop(env, task, bad_plan, ground_truth)

## ORM 和 PRM

可以注意到：
1. immediate 是 reward 模型给每一个步骤的奖励（ORM是只对结果有反应，所以中间过程都是0）
2. immediate[t] + gamma * G 这个公式是折扣累计回报

In [ ]:
# ==========================================
# 4. ORM vs PRM 信用分配
# ==========================================
import numpy as np

def orm_credit(episode: Episode, gamma: float = 0.95) -> List[float]:
    """ORM：只有最终结果给 reward，中间步骤全部为 0。"""
    T = len(episode.turns)
    final_success = episode.turns[-1].success
    immediate = [0.0] * (T - 1) + [1.0 if final_success else -1.0]

    returns = np.zeros(T)
    G = 0
    for t in reversed(range(T)):
        G = immediate[t] + gamma * G
        returns[t] = G
    return returns.tolist()

def prm_credit(episode: Episode, gamma: float = 0.95) -> List[float]:
    """PRM：每一步根据工具调用是否成功给即时 reward。"""
    T = len(episode.turns)
    immediate = []
    for turn in episode.turns:
        if turn.action == "answer":
            immediate.append(1.0 if turn.success else -0.5)
        else:
            immediate.append(0.3 if turn.success else -0.3)

    returns = np.zeros(T)
    G = 0
    for t in reversed(range(T)):
        G = immediate[t] + gamma * G
        returns[t] = G
    return returns.tolist()

orm_bad = orm_credit(bad_episode)
prm_bad = prm_credit(bad_episode)

# 对于坏动作序列， 分别打印ORM和PRM的信用分配
print(f"\n{'轮次':<6} {'动作':<12} {'结果':<8} {'ORM Credit':<14} {'PRM Credit':<14}")
for i, turn in enumerate(bad_episode.turns):
    status = "✓" if turn.success else "✗"
    print(f"第{i+1}轮   {turn.action:<12} {status:<8} {orm_bad[i]:<14.3f} {prm_bad[i]:<14.3f}")

NameError: name 'Episode' is not defined

这里就会发现在orm下， 即便第一步正确的步骤，由于最后没有成功， 也会给负数reward，所以ORM